In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Đánh giá full RAG + Critic Agent trên Colab

Chỉ cần sửa tham số ở Cell **THAM SỐ** bên dưới, rồi **Runtime → Run all**.

**Yêu cầu trước khi chạy:**
1. Runtime → Change runtime type → chọn **GPU** (T4 miễn phí là đủ).
2. Đã upload `data/.qdrant/` và `data/ai_vietnamese_embedding_v2_finetuned_final/` lên Google Drive (đúng thư mục điền ở `DRIVE_DATA_DIR`).
3. Đã đưa Knowledge Graph lên Neo4j Aura (điền `NEO4J_URI`/`NEO4J_PASSWORD` đúng của bạn).


In [ ]:
# ============================================================
# THAM SỐ — CHỈNH Ở ĐÂY, KHÔNG CẦN SỬA GÌ Ở CÁC Ô BÊN DƯỚI
# ============================================================

# --- Neo4j Aura (điền thông tin của bạn — KHÔNG commit lại giá trị thật vào git) ---
NEO4J_URI = "neo4j+s://xxxxxxxx.databases.neo4j.io"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = ""

# --- Google Drive: nơi đã upload sẵn data/.qdrant/ và model fine-tune ---
DRIVE_DATA_DIR = "/content/drive/MyDrive/legal_kg_data"

# --- So sánh với embedding GỐC (chưa fine-tune) — để trống 2 dòng dưới = dùng bản
# fine-tune mặc định của repo, không cần đổi gì. Điền vào để chạy so sánh, KHÔNG
# cần sửa code (build_pipeline() trong scripts/run_chatbot.py tự đọc 2 biến môi
# trường QDRANT_PATH/EMBEDDING_MODEL tương ứng 2 tham số này). ĐỔI OUTPUT_SUFFIX
# khi chạy so sánh (vd "_baseembed301") để không ghi đè kết quả bản fine-tune.
EMBEDDING_MODEL_OVERRIDE = ""   # vd "AITeamVN/Vietnamese_Embedding_v2" (model gốc, tự tải từ HuggingFace, không cần copy từ Drive)
QDRANT_SUBDIR_OVERRIDE = ""     # vd ".qdrant_base" — tên thư mục Qdrant TƯƠNG ỨNG đã ingest sẵn bằng model gốc, nằm trong DRIVE_DATA_DIR (PHẢI ingest trước bằng src/data_ingestion/qdrant_local_ingest.py --model ... --db-path data/.qdrant_base rồi upload thư mục này lên Drive — KHÔNG dùng chung .qdrant với bản fine-tune vì 2 model có không gian embedding khác nhau)

# --- Git repo chứa code ---
GIT_REPO_URL = "https://github.com/PhuIT2503/LEGAL_IT_CHATBOT.git"
REPO_DIR = "LEGAL_IT_CHATBOT"

# --- LLM sinh câu trả lời (Ollama, chạy ngay trong Colab, dùng GPU) ---
OLLAMA_MODEL = "qwen2.5:7b"

# --- Bộ câu hỏi test ---
# vd: "data/eval_testset.jsonl"              (301 câu, đủ 4 nhóm)
#     "data/eval_testset_stratified10.jsonl" (10 câu, nhanh để test thử)
TESTSET_PATH = "data/eval_testset.jsonl"
LIMIT = None                 # None = chạy hết; hoặc số nguyên (vd 10) để test nhanh
OUTPUT_SUFFIX = "_full301"   # hậu tố tên file kết quả — đổi mỗi lần chạy để không ghi đè
MODES = ["naive", "article_expand", "critic"]
RESUME = True                # True = bỏ qua câu đã có sẵn kết quả (chạy lại an toàn sau khi bị ngắt)

# --- LLM DUY NHẤT dùng để CHẤM ĐIỂM (CẢ Completeness Rate VÀ RAGAS) ---
# Tách biệt khỏi Ollama/qwen (model SINH câu trả lời ở trên) để tránh
# self-preference bias — model không tự chấm câu trả lời của chính họ.
# gpt-4o-mini: rẻ (~1-3 đô cho vài trăm câu CẢ 2 lớp chỉ số), đáng tin cậy,
# không rủi ro bị khóa account (KHÔNG dùng nhiều key xoay vòng để lách limit):
JUDGE_PROVIDER = "openai"    # "openai" | "gemini" | "ollama"  (ollama = free, không cần key, nhưng có rủi ro self-bias)
JUDGE_MODEL = "gpt-4o-mini"  # mặc định của provider (gpt-4o-mini / gemini-1.5-flash / qwen2.5:7b)
OPENAI_API_KEY = ""          # chỉ cần điền nếu JUDGE_PROVIDER == "openai"
GOOGLE_API_KEY = ""          # chỉ cần điền nếu JUDGE_PROVIDER == "gemini"

# --- RAGAS (faithfulness / answer_relevancy / context_precision / answer_correctness) ---
# Lớp chỉ số THÊM, tùy chọn — dùng CHUNG model chấm ở trên (JUDGE_PROVIDER).
SKIP_RAGAS = False            # True nếu chỉ muốn Completeness Rate, bỏ qua RAGAS hoàn toàn

# RAGAS chấm theo batch + lưu checkpoint (data/ragas_checkpoint_<mode><suffix>.json)
# sau MỖI batch — file này được cell auto-backup (bên dưới) đồng bộ lên Drive
# mỗi 60s, nên nếu bị ngắt phiên Colab, chạy lại từ đầu vẫn khôi phục được
# checkpoint từ Drive và chấm tiếp, KHÔNG mất tiền/thời gian chấm lại.
# Completeness Rate cũng lưu checkpoint TỪNG CÂU tương tự
# (data/completeness_checkpoint_<mode><suffix>.json), cùng cơ chế backup/restore.
RAGAS_BATCH_SIZE = 10

# Mỗi job RAGAS lỗi (rate limit, timeout) chỉ thử lại tối đa RAGAS_MAX_RETRIES lần
# rồi bỏ qua (ragas mặc định gốc là 10 lần — dễ 'treo' rất lâu nếu API đang bị giới
# hạn kéo dài). Câu bị bỏ qua vẫn tự được chấm lại ở lần chạy SAU nhờ checkpoint.
RAGAS_MAX_RETRIES = 2


## 1. Mount Google Drive

## 2. Lấy code từ GitHub

In [ ]:
import os

# Luôn neo về /content trước khi kiểm tra — tránh bug clone lồng nhau nếu
# cell này được chạy lại lần 2 (lúc đó cwd đã nằm trong REPO_DIR từ lần trước).
os.chdir("/content")

if not os.path.isdir(REPO_DIR):
    !git clone {GIT_REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} đã tồn tại, pull code mới nhất...")
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}


## 3. Copy dữ liệu nặng (.qdrant + model fine-tune) từ Drive

In [ ]:
import os

os.makedirs("data", exist_ok=True)

qdrant_subdir = QDRANT_SUBDIR_OVERRIDE or ".qdrant"
qdrant_src = os.path.join(DRIVE_DATA_DIR, qdrant_subdir)
qdrant_dest = os.path.join("data", qdrant_subdir)

if not os.path.isdir(qdrant_dest):
    print(f"Copy data/{qdrant_subdir} từ Drive...")
    !cp -r "{qdrant_src}" "{qdrant_dest}"
else:
    print(f"data/{qdrant_subdir} đã có sẵn, bỏ qua copy.")

if not EMBEDDING_MODEL_OVERRIDE:
    # Bản fine-tune cục bộ (file model lớn, chỉ có trên Drive của bạn) — PHẢI copy.
    model_src = os.path.join(DRIVE_DATA_DIR, "ai_vietnamese_embedding_v2_finetuned_final")
    if not os.path.isdir("data/ai_vietnamese_embedding_v2_finetuned_final"):
        print("Copy model fine-tune từ Drive (~2.2GB, có thể mất vài phút)...")
        !cp -r "{model_src}" data/ai_vietnamese_embedding_v2_finetuned_final
    else:
        print("Model fine-tune đã có sẵn, bỏ qua copy.")
else:
    # Model gốc (vd AITeamVN/Vietnamese_Embedding_v2) là model công khai trên
    # HuggingFace — KHÔNG cần copy từ Drive, tự tải khi dùng lần đầu.
    print(f"Dùng embedding '{EMBEDDING_MODEL_OVERRIDE}' — tự tải từ HuggingFace khi cần, không copy từ Drive.")

print("Xong.")

## 4. Cài dependencies

In [ ]:
!pip install -q -r requirements.txt "langchain-core<0.3.0" "langgraph<0.3.0" "langchain-openai<0.2.0" "langchain-google-genai<2.0.0"
!pip install -q ragas==0.1.22 datasets


## 5. Cài + chạy Ollama (dùng GPU Colab)

In [ ]:
import subprocess, time, requests

# Cảnh báo sớm nếu quên chọn GPU (Runtime > Change runtime type > GPU) —
# Ollama vẫn chạy được trên CPU nhưng chậm hơn nhiều (~5-10x).
gpu_check = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_check.returncode != 0:
    print("⚠️  KHÔNG phát hiện GPU — vào Runtime > Change runtime type > chọn GPU rồi chạy lại từ đầu để tốc độ sinh câu trả lời nhanh hơn nhiều.")
else:
    print("✓ GPU sẵn sàng.")

!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

ollama_proc = subprocess.Popen(["ollama", "serve"])

ollama_ready = False
for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        ollama_ready = True
        break
    except Exception:
        time.sleep(2)

if not ollama_ready:
    print("⚠️  Ollama chưa phản hồi sau 60s — cell tiếp theo (ollama pull) có thể lỗi, thử chạy lại cell này.")

!ollama pull {OLLAMA_MODEL}
print("Ollama sẵn sàng.")


## 6. Set biến môi trường (Aura + RAGAS key nếu có)

In [ ]:
import os

os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USER"] = NEO4J_USER
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD
os.environ["OLLAMA_MODEL"] = OLLAMA_MODEL

qdrant_subdir = QDRANT_SUBDIR_OVERRIDE or ".qdrant"
os.environ["QDRANT_PATH"] = f"data/{qdrant_subdir}"
if EMBEDDING_MODEL_OVERRIDE:
    os.environ["EMBEDDING_MODEL"] = EMBEDDING_MODEL_OVERRIDE

if JUDGE_PROVIDER == "openai" and OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
if JUDGE_PROVIDER == "gemini" and GOOGLE_API_KEY:
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print("Đã set biến môi trường.")


In [ ]:
import shutil, glob, os

# Thư mục chứa file đang lưu dở trên Drive
src_dir = os.path.join(DRIVE_DATA_DIR, "results")
dest_dir = "data"
os.makedirs(dest_dir, exist_ok=True)

# Lấy lại file kết quả VÀ checkpoint RAGAS đang lưu dở (nếu có) về máy ảo —
# nhờ vậy nếu phiên Colab trước bị ngắt giữa lúc chấm RAGAS, lần chạy này
# vẫn tiếp tục đúng từ checkpoint, không tốn tiền/thời gian chấm lại.
patterns = [
    f"{src_dir}/eval_results_*{OUTPUT_SUFFIX}.jsonl",
    f"{src_dir}/ragas_checkpoint_*{OUTPUT_SUFFIX}.json",
    f"{src_dir}/completeness_checkpoint_*{OUTPUT_SUFFIX}.json",
]
for pattern in patterns:
    for f in glob.glob(pattern):
        shutil.copy(f, dest_dir)
        print(f"✅ Đã khôi phục file chạy dở: {f}")


In [ ]:
import threading
import time
import shutil
import glob
import os

def backup_to_drive_continuously():
    dest = os.path.join(DRIVE_DATA_DIR, "results")
    os.makedirs(dest, exist_ok=True)
    while True:
        patterns = [
            f"data/eval_results_*{OUTPUT_SUFFIX}.jsonl",
            f"data/eval_scores_*{OUTPUT_SUFFIX}.csv",
            f"data/eval_summary{OUTPUT_SUFFIX}.csv",
            f"data/ragas_checkpoint_*{OUTPUT_SUFFIX}.json",
            f"data/completeness_checkpoint_*{OUTPUT_SUFFIX}.json",
        ]
        for pattern in patterns:
            for f in glob.glob(pattern):
                try:
                    shutil.copy(f, dest)
                except:
                    pass
        time.sleep(60)

t = threading.Thread(target=backup_to_drive_continuously, daemon=True)
t.start()
print("✅ Đã bật chế độ tự động sao lưu kết quả + checkpoint RAGAS/Completeness Rate lên Drive mỗi 60 giây!")


## 7. Chạy đánh giá (run_evaluation.py) cho từng mode

In [ ]:
for mode in MODES:
    cmd = f'python scripts/run_evaluation.py --mode {mode} --testset "{TESTSET_PATH}"'
    if OUTPUT_SUFFIX:
        cmd += f' --output-suffix "{OUTPUT_SUFFIX}"'
    if LIMIT:
        cmd += f' --limit {LIMIT}'
    if RESUME:
        cmd += ' --resume'
    print(f"\n{'='*80}\n=== Chạy mode={mode} ===\n{'='*80}")
    get_ipython().system(cmd)


In [ ]:
%%writefile scripts/score_evaluation.py
"""
scripts/score_evaluation.py
=============================
Tính điểm cho kết quả đã chạy (data/eval_results_<mode>.jsonl, xem
run_evaluation.py) — gồm 2 lớp chỉ số:

1. RAGAS chuẩn (nếu đã `pip install ragas`): faithfulness, answer_relevancy,
   context_precision, answer_correctness (cần `reference` — đã có sẵn trong
   test set). Bọc trong try/except: nếu chưa cài ragas hoặc lệch API version,
   script vẫn chạy tiếp phần chỉ số tùy biến bên dưới, không crash toàn bộ.

2. Legal Completeness Rate (tùy biến, LLM-as-judge) — chỉ số TRUNG TÂM của
   khóa luận: với mỗi fact trong required_facts của từng câu hỏi, hỏi LLM xem
   câu trả lời (response) có thể hiện đúng nội dung fact đó không. Tỷ lệ fact
   được thể hiện đúng = Legal Completeness Rate của câu đó.

Cả 2 lớp chỉ số dùng CHUNG đúng 1 LLM chấm (--judge-provider/--judge-model,
xem build_llm_for_provider()) — đo nhất quán, và tách biệt khỏi model SINH
câu trả lời (Ollama/qwen của cả 3 kịch bản) để tránh self-preference bias.

Output: in bảng tổng hợp theo mode x category, lưu CSV chi tiết từng câu tại
data/eval_scores_<mode>.csv và bảng tổng hợp tại data/eval_summary.csv.

Cách dùng:
    python scripts/score_evaluation.py --modes naive article_expand critic
    python scripts/score_evaluation.py --modes critic --skip-ragas   # chỉ tính Completeness Rate, bỏ qua RAGAS

    # Tự do chọn LLM chấm điểm (CẢ Completeness Rate VÀ RAGAS), KHÔNG hardcode:
    python scripts/score_evaluation.py --modes critic --judge-provider openai --judge-model gpt-4o-mini
        (cần set OPENAI_API_KEY trong môi trường — rẻ, ~vài đô cho vài trăm câu)
    python scripts/score_evaluation.py --modes critic --judge-provider gemini --judge-model gemini-1.5-flash
        (cần set GOOGLE_API_KEY trong môi trường)
    python scripts/score_evaluation.py --modes critic --judge-provider ollama
        (dùng lại Ollama local đang chạy sẵn — miễn phí, không cần API key,
        nhưng có rủi ro self-preference bias vì đây cũng là model sinh câu
        trả lời của cả 3 kịch bản)

    RAGAS mặc định chấm theo BATCH (10 câu/lần, xem --ragas-batch-size) và lưu
    checkpoint tại data/ragas_checkpoint_<mode><suffix>.json sau mỗi batch —
    nếu bị ngắt giữa chừng (mất mạng, hết quota...), chạy lại ĐÚNG lệnh cũ sẽ
    tự động chấm tiếp phần còn thiếu, không mất tiền/thời gian chấm lại từ đầu.
    Dùng --no-ragas-checkpoint để tắt, chấm 1 lần nguyên khối như bản cũ.

    Legal Completeness Rate cũng lưu checkpoint TỪNG CÂU tại
    data/completeness_checkpoint_<mode><suffix>.json ngay sau khi chấm xong —
    cùng cơ chế resume như RAGAS ở trên. Dùng --no-completeness-checkpoint để
    tắt.

    Mỗi job RAGAS lỗi (rate limit, timeout) mặc định chỉ thử lại 2 lần rồi bỏ
    qua (xem --ragas-max-retries) thay vì thử tới 10 lần như mặc định gốc của
    ragas — tránh "treo" rất lâu khi API đang bị giới hạn kéo dài; câu bị bỏ
    qua vẫn tự được chấm lại ở lần chạy SAU nhờ checkpoint.
"""
import argparse
import csv
import json
import os
import sys
from collections import defaultdict
from pathlib import Path
from typing import List

PROJECT_ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from langchain_openai import ChatOpenAI  # noqa: E402
from langchain_core.embeddings import Embeddings as _LangchainEmbeddingsBase  # noqa: E402


def build_llm_for_provider(provider: str, model_name: str = None):
    """
    Dựng 1 LLM DUY NHẤT (temperature=0) — dùng CHUNG cho cả 2 lớp chỉ số
    (Legal Completeness Rate VÀ RAGAS), để đo bằng đúng 1 model nhất quán,
    không lệch chuẩn giữa 2 lớp khi đưa số liệu vào khóa luận.

    Trước đây Completeness Rate luôn dùng cứng Ollama/qwen (model SINH câu trả
    lời của cả 3 kịch bản) để chấm — có rủi ro lý thuyết "self-preference
    bias" (model có xu hướng tự chấm câu trả lời của chính nó/model cùng họ
    cao hơn thực tế). Gộp về 1 provider tự chọn (mặc định openai/gpt-4o-mini,
    độc lập với model sinh câu trả lời) loại bỏ rủi ro này, và không tốn thêm
    đáng kể (mỗi lệnh gọi chấm 1 fact rất ngắn).

    temperature=0 (KHÁC với LLM sinh câu trả lời, temperature=0.2) vì đây là
    tác vụ PHÂN LOẠI yes/no đơn giản — quan sát thực tế: CÙNG 1 câu trả lời
    (chữ giống hệt nhau giữa 2 mode) nhưng judge chấm khác nhau giữa các lần
    chạy nếu để temperature>0 — nhiễu ngẫu nhiên của chính bước chấm, không
    phải khác biệt chất lượng thật.

      - "openai": ChatOpenAI thật, cần OPENAI_API_KEY trong môi trường.
      - "gemini": ChatGoogleGenerativeAI, cần GOOGLE_API_KEY trong môi trường.
      - "ollama": Ollama local đang chạy sẵn — miễn phí, không cần API key.
    """
    if provider == "openai":
        if not os.getenv("OPENAI_API_KEY"):
            raise RuntimeError("--judge-provider openai cần biến môi trường OPENAI_API_KEY.")
        return ChatOpenAI(model=model_name or "gpt-4o-mini", temperature=0.0)
    elif provider == "gemini":
        if not os.getenv("GOOGLE_API_KEY"):
            raise RuntimeError("--judge-provider gemini cần biến môi trường GOOGLE_API_KEY.")
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model=model_name or "gemini-1.5-flash", temperature=0.0)
    elif provider == "ollama":
        base_url = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")
        model = model_name or os.getenv("OLLAMA_MODEL", "qwen2.5:7b")
        return ChatOpenAI(model=model, base_url=base_url, api_key="ollama", temperature=0.0)
    else:
        raise ValueError(f"--judge-provider không hợp lệ: {provider!r} (chọn openai|gemini|ollama)")


# Tên hiển thị RÕ RÀNG cho từng chỉ số RAGAS khi in ra console — key thật trả
# về từ ragas.evaluate() là tên ngắn (faithfulness, answer_relevancy, ...).
RAGAS_METRIC_LABELS = {
    "faithfulness": "Faithfulness (độ trung thực với ngữ cảnh)",
    "answer_relevancy": "Answer Relevancy (độ liên quan của câu trả lời)",
    "context_precision": "Context Precision (độ chính xác ngữ cảnh, theo RAGAS)",
    "answer_correctness": "Answer Correctness (độ đúng đắn câu trả lời)",
}


class _LocalSentenceTransformerEmbeddings(_LangchainEmbeddingsBase):
    """
    Wrapper Embeddings kiểu LangChain (embed_documents/embed_query) quanh
    SentenceTransformer cục bộ (model fine-tune VBPL) — dùng cho RAGAS thay vì
    bắt buộc phải có thêm 1 API key riêng cho embeddings. Dùng CHUNG cho mọi
    provider LLM (openai/gemini/ollama) vì embeddings và LLM chấm là 2 việc
    độc lập trong RAGAS.

    PHẢI kế thừa langchain_core.embeddings.Embeddings (không phải object
    thường) — RAGAS chấm bất đồng bộ (asyncio) và cần aembed_documents/
    aembed_query, mà class cha này tự cung cấp bản async mặc định (chạy
    embed_documents/embed_query đồng bộ qua thread executor). Thiếu bước kế
    thừa này thì mọi lệnh gọi async đều lỗi AttributeError, khiến các chỉ số
    phụ thuộc embedding (answer_relevancy, answer_correctness) bị NaN/thiếu
    dữ liệu ở một phần câu hỏi mà không crash toàn bộ — rất dễ bị bỏ sót.
    """

    def __init__(self, model):
        self._model = model

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return self._model.encode(list(texts), normalize_embeddings=True).tolist()

    def embed_query(self, text: str) -> List[float]:
        return self._model.encode([text], normalize_embeddings=True)[0].tolist()


def build_ragas_llm_and_embeddings(provider: str, model_name: str = None):
    """
    Dựng (llm, embeddings) cho RAGAS — dùng lại ĐÚNG build_llm_for_provider()
    (cùng 1 model với Completeness Rate, xem docstring hàm đó). Embeddings mặc
    định THEO ĐÚNG embedding đang dùng cho retrieval (biến môi trường
    EMBEDDING_MODEL — xem build_pipeline() trong scripts/run_chatbot.py), để
    nhất quán khi so sánh fine-tune vs embedding gốc — set RAGAS_EMBEDDING_MODEL
    riêng nếu thật sự muốn RAGAS dùng embedding KHÁC với retrieval.
    """
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from src.llm.embedding_model import load_embedding_model

    llm = build_llm_for_provider(provider, model_name)

    embed_model = load_embedding_model(
        os.getenv("RAGAS_EMBEDDING_MODEL") or os.getenv("EMBEDDING_MODEL", "data/ai_vietnamese_embedding_v2_finetuned_final")
    )
    embeddings = LangchainEmbeddingsWrapper(_LocalSentenceTransformerEmbeddings(embed_model))
    return LangchainLLMWrapper(llm), embeddings


def load_results(mode: str, suffix: str = ""):
    path = PROJECT_ROOT / "data" / f"eval_results_{mode}{suffix}.jsonl"
    if not path.exists():
        print(f"CẢNH BÁO: chưa có {path} — hãy chạy run_evaluation.py --mode {mode} trước.")
        return []
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def judge_fact_covered(llm, response: str, fact: str) -> bool:
    """LLM-as-judge: fact có được thể hiện ĐÚNG trong response hay không (yes/no)."""
    prompt = (
        "Bạn là giám khảo chấm điểm câu trả lời pháp luật. Cho một CÂU TRẢ LỜI và một YÊU CẦU "
        "(1 fact bắt buộc phải có), hãy xác định xem CÂU TRẢ LỜI có thể hiện ĐÚNG nội dung của YÊU CẦU "
        "hay không — chấp nhận diễn đạt khác nhau miễn là ĐÚNG Ý và ĐÚNG SỐ LIỆU cụ thể (nếu yêu cầu có số "
        "liệu). Nếu câu trả lời thiếu hẳn ý đó, diễn đạt mơ hồ né tránh, hoặc nêu sai số liệu/nội dung thì "
        "tính là KHÔNG đạt.\n\n"
        f"YÊU CẦU (fact bắt buộc): {fact}\n\n"
        f"CÂU TRẢ LỜI CẦN CHẤM:\n{response}\n\n"
        "Chỉ trả lời đúng 1 từ: 'yes' nếu câu trả lời có thể hiện đúng fact này, 'no' nếu không."
    )
    resp = llm.invoke(prompt)
    return "yes" in resp.content.strip().lower()


def compute_completeness(llm, rows: list, checkpoint_path: Path = None) -> list:
    """Gắn thêm completeness_rate + facts_covered vào từng row (mutate + return).

    checkpoint_path: nếu có, lưu kết quả từng câu (theo id) ra file JSON ngay
    sau khi chấm xong câu đó — nếu bị ngắt giữa chừng (mất mạng, hết quota,
    Colab bị disconnect...), chạy lại đúng lệnh cũ sẽ chỉ chấm tiếp các câu
    CHƯA có trong checkpoint, không mất tiền/thời gian chấm lại từ đầu. Không
    truyền checkpoint_path -> hành vi cũ, không lưu tạm, mất là mất hết.
    """
    per_row = _load_json_checkpoint(checkpoint_path) if checkpoint_path else {}
    if per_row:
        print(f"  Đã nạp checkpoint Completeness Rate: {len(per_row)}/{len(rows)} câu đã chấm từ lần chạy trước.")

    todo_count = sum(1 for r in rows if r["id"] not in per_row)
    done_count = 0
    for row in rows:
        cached = per_row.get(row["id"])
        if cached is not None:
            row["completeness_rate"] = cached.get("completeness_rate")
            row["facts_covered"] = cached.get("facts_covered")
            continue

        facts = row.get("required_facts", [])
        if not facts:
            row["completeness_rate"] = None
            row["facts_covered"] = None
        else:
            covered = [judge_fact_covered(llm, row["response"], f) for f in facts]
            row["facts_covered"] = covered
            row["completeness_rate"] = sum(covered) / len(covered)

        per_row[row["id"]] = {"completeness_rate": row["completeness_rate"], "facts_covered": row["facts_covered"]}
        done_count += 1
        if checkpoint_path is not None:
            _save_json_checkpoint(checkpoint_path, per_row)
            if done_count % 20 == 0 or done_count == todo_count:
                print(f"  Completeness Rate: đã chấm {done_count}/{todo_count} câu mới ({len(per_row)}/{len(rows)} tổng cộng)...")
    return rows


def _load_json_checkpoint(checkpoint_path: Path) -> dict:
    if not checkpoint_path.exists():
        return {}
    with open(checkpoint_path, encoding="utf-8") as f:
        return json.load(f)


def _save_json_checkpoint(checkpoint_path: Path, data: dict) -> None:
    tmp_path = checkpoint_path.with_suffix(".tmp")
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
    tmp_path.replace(checkpoint_path)


def try_compute_ragas(
    rows: list,
    ragas_provider: str,
    ragas_model: str = None,
    checkpoint_path: Path = None,
    batch_size: int = 10,
    max_retries: int = 2,
) -> dict:
    """Trả về dict {metric_name: avg_score} nếu ragas cài được và chạy được, ngược lại {}.

    ragas_provider: "openai" | "gemini" | "ollama" — xem
    build_ragas_llm_and_embeddings() và docstring đầu file để biết cách
    truyền API key cho từng provider.

    checkpoint_path: nếu có, chấm theo TỪNG BATCH (batch_size câu/lần) và lưu
    điểm từng câu (theo id) ra file JSON sau MỖI batch — nếu bị ngắt giữa
    chừng (mất mạng, hết quota, Colab bị disconnect...), chạy lại đúng lệnh
    cũ sẽ chỉ chấm tiếp phần CHƯA ĐỦ 4 chỉ số trong checkpoint, không mất
    tiền/thời gian chấm lại từ đầu. Không truyền checkpoint_path -> chấm 1
    lần nguyên khối như cũ (không lưu tạm, mất là mất hết).

    Một câu được coi "xong" chỉ khi checkpoint có ĐỦ CẢ 4 chỉ số — vì trong
    1 batch, evaluate() có thể chấm THÀNH CÔNG cho batch nhưng vẫn thiếu 1-2
    chỉ số ở MỘT VÀI câu do lỗi tạm thời ở đúng job đó (rate limit, timeout —
    xem comment ở batch_ok bên dưới). Nếu chỉ kiểm tra "câu đã có trong
    checkpoint" mà không kiểm tra đủ chỉ số, các câu thiếu 1 phần này sẽ bị
    bỏ sót vĩnh viễn, không bao giờ được chấm lại dù resume bao nhiêu lần.

    max_retries: ragas mặc định thử lại TỚI 10 LẦN mỗi job lỗi (RunConfig mặc
    định: max_retries=10, max_wait=60s, timeout=180s/lần thử) — 1 job cứ lỗi
    hoài (vd đang bị rate limit kéo dài) có thể "treo" nhiều phút trước khi
    ragas chịu bỏ cuộc, nhân với hàng chục job/batch thành rất lâu. Giảm
    xuống số lần thử thấp (mặc định 2) để job lỗi được bỏ qua NHANH trong
    lần chạy này — vẫn không mất gì vì cơ chế resume ở trên sẽ tự chấm lại
    đúng câu đó ở lần chạy SAU.
    """
    try:
        from datasets import Dataset
        from ragas import evaluate
        from ragas.metrics import faithfulness, answer_relevancy, context_precision, answer_correctness
        from ragas.run_config import RunConfig
    except ImportError as e:
        print(f"Bỏ qua RAGAS (chưa cài đặt hoặc thiếu dependency): {e}")
        print("Cài đặt: pip install ragas datasets")
        return {}

    run_config = RunConfig(max_retries=max_retries, max_wait=10, timeout=60)

    try:
        llm, embeddings = build_ragas_llm_and_embeddings(ragas_provider, ragas_model)
    except Exception as e:
        print(f"Bỏ qua RAGAS (không dựng được LLM/embeddings cho provider={ragas_provider!r}): {e}")
        return {}

    metrics = [faithfulness, answer_relevancy, context_precision, answer_correctness]
    metric_names = [m.name for m in metrics]

    def _to_ragas_item(r):
        return {
            "question": r["user_input"],
            "answer": r["response"],
            "contexts": r["retrieved_contexts"] or [""],
            "ground_truth": r["reference"],
        }

    if checkpoint_path is None:
        # Hành vi CŨ — chấm 1 lần nguyên khối, không lưu tạm.
        ds = Dataset.from_list([_to_ragas_item(r) for r in rows])
        try:
            result = evaluate(ds, metrics=metrics, llm=llm, embeddings=embeddings, run_config=run_config)
            return {k: float(v) for k, v in result.items()}
        except Exception as e:
            print(f"RAGAS evaluate() lỗi (có thể do khác version API) — báo cáo lỗi để tự điều chỉnh: {e}")
            return {}

    per_row_scores = _load_json_checkpoint(checkpoint_path)
    if per_row_scores:
        complete = sum(1 for s in per_row_scores.values() if len(s) >= len(metric_names))
        print(f"  Đã nạp checkpoint RAGAS: {len(per_row_scores)}/{len(rows)} câu có trong checkpoint "
              f"({complete} câu đủ cả {len(metric_names)} chỉ số, {len(per_row_scores) - complete} câu thiếu "
              f"một phần do lỗi tạm thời trước đó — sẽ chấm lại các câu thiếu này).")

    # Coi 1 câu là "xong" chỉ khi ĐỦ cả 4 chỉ số — nếu batch trước bị rate limit/
    # timeout giữa chừng, checkpoint có thể lưu câu đó với chỉ 1-3/4 chỉ số (xem
    # comment ở dưới). Nếu chỉ kiểm tra "đã có trong checkpoint" (không kiểm tra đủ
    # chỉ số), các câu thiếu 1 phần này sẽ bị coi là xong VĨNH VIỄN, không bao giờ
    # được chấm lại dù chạy lại bao nhiêu lần.
    todo_rows = [r for r in rows if len(per_row_scores.get(r["id"], {})) < len(metric_names)]
    consecutive_bad_batches = 0
    for i in range(0, len(todo_rows), batch_size):
        batch = todo_rows[i : i + batch_size]
        ds = Dataset.from_list([_to_ragas_item(r) for r in batch])
        try:
            result = evaluate(ds, metrics=metrics, llm=llm, embeddings=embeddings, run_config=run_config)
            df = result.to_pandas()
        except Exception as e:
            print(f"  RAGAS lỗi ở batch {i}-{i + len(batch)}: {e}")
            print(f"  Đã lưu checkpoint tới {len(per_row_scores)}/{len(rows)} câu — chạy lại lệnh cũ để tiếp tục.")
            break

        # LƯU Ý: evaluate() KHÔNG raise exception khi từng job con lỗi (RateLimitError,
        # TimeoutError...) — ragas tự bắt lỗi ở mức job, chỉ in "Exception raised in
        # Job[N]" và trả về NaN cho đúng ô đó, evaluate() vẫn coi là "thành công". Vì
        # vậy try/except phía trên KHÔNG bắt được tình trạng hết quota API (vd RPD của
        # OpenAI cạn) — phải tự đếm tỷ lệ NaN mỗi batch để phát hiện và DỪNG kịp thời,
        # nếu không sẽ chạy hết batch còn lại (có thể hàng giờ) mà toàn ra dữ liệu rỗng.
        batch_ok = 0
        for row, (_, score_row) in zip(batch, df.iterrows()):
            scores = {}
            for name in metric_names:
                val = score_row.get(name)
                if val is not None and val == val:  # loại NaN (NaN != NaN)
                    scores[name] = float(val)
                    batch_ok += 1
            per_row_scores[row["id"]] = scores
        _save_json_checkpoint(checkpoint_path, per_row_scores)

        batch_total = len(batch) * len(metric_names)
        success_rate = batch_ok / batch_total if batch_total else 1.0
        print(f"  RAGAS: đã chấm {min(i + batch_size, len(todo_rows))}/{len(todo_rows)} câu mới "
              f"({len(per_row_scores)}/{len(rows)} tổng cộng, batch này thành công {success_rate:.0%})...")

        if success_rate < 0.5:
            consecutive_bad_batches += 1
        else:
            consecutive_bad_batches = 0

        if consecutive_bad_batches >= 2:
            print(f"  DỪNG: 2 batch liên tiếp thất bại phần lớn (khả năng hết quota/rate limit API, xem "
                  f"'Exception raised in Job[...]' ở log phía trên). Đã lưu checkpoint tới "
                  f"{len(per_row_scores)}/{len(rows)} câu — chạy lại ĐÚNG lệnh cũ sau khi hết bị rate limit "
                  f"để tự động chấm tiếp phần còn thiếu, không mất tiền/thời gian chấm lại.")
            break

    if not per_row_scores:
        return {}
    agg = {}
    for name in metric_names:
        vals = [per_row_scores[r["id"]][name] for r in rows if name in per_row_scores.get(r["id"], {})]
        if vals:
            agg[name] = sum(vals) / len(vals)
    return agg


def main():
    parser = argparse.ArgumentParser(description="Tính điểm đánh giá (RAGAS + Legal Completeness Rate tùy biến)")
    parser.add_argument("--modes", nargs="+", default=["naive", "article_expand", "critic"])
    parser.add_argument("--skip-ragas", action="store_true")
    parser.add_argument("--suffix", type=str, default="", help="Hậu tố file input/output (vd '_stratified10'), phải khớp với --output-suffix đã dùng ở run_evaluation.py")
    parser.add_argument("--judge-provider", choices=["openai", "gemini", "ollama"], default="openai",
                         help="LLM DUY NHẤT dùng để chấm CẢ Completeness Rate VÀ RAGAS — xem docstring build_llm_for_provider() để biết API key cần set cho từng provider")
    parser.add_argument("--judge-model", type=str, default=None,
                         help="Tên model cụ thể cho --judge-provider (vd gpt-4o-mini, gemini-1.5-flash, qwen2.5:7b). Bỏ trống dùng mặc định của provider.")
    parser.add_argument("--ragas-batch-size", type=int, default=10,
                         help="Số câu chấm RAGAS mỗi batch trước khi lưu checkpoint (mặc định 10) — batch nhỏ hơn = lưu thường xuyên hơn, an toàn hơn nếu hay bị ngắt, nhưng chậm hơn 1 chút.")
    parser.add_argument("--ragas-max-retries", type=int, default=2,
                         help="Số lần ragas thử lại mỗi job lỗi trước khi bỏ qua (mặc định 2, ragas mặc định gốc là 10 — dễ 'treo' rất lâu nếu đang bị rate limit kéo dài). Câu bị bỏ qua vẫn được chấm lại ở lần chạy SAU nhờ checkpoint, không mất gì.")
    parser.add_argument("--no-ragas-checkpoint", action="store_true",
                         help="Tắt checkpoint, chấm RAGAS 1 lần nguyên khối như bản cũ (mất là mất hết nếu bị ngắt giữa chừng).")
    parser.add_argument("--no-completeness-checkpoint", action="store_true",
                         help="Tắt checkpoint, chấm Completeness Rate lại từ đầu mỗi lần (mất là mất hết nếu bị ngắt giữa chừng).")
    args = parser.parse_args()

    llm = build_llm_for_provider(args.judge_provider, args.judge_model)

    summary_rows = []
    for mode in args.modes:
        rows = load_results(mode, args.suffix)
        if not rows:
            continue

        print(f"\n=== Chấm điểm mode={mode} ({len(rows)} câu) ===")
        completeness_ckpt_path = None
        if not args.no_completeness_checkpoint:
            completeness_ckpt_path = PROJECT_ROOT / "data" / f"completeness_checkpoint_{mode}{args.suffix}.json"
        rows = compute_completeness(llm, rows, completeness_ckpt_path)

        ragas_scores = {}
        if not args.skip_ragas:
            ckpt_path = None
            if not args.no_ragas_checkpoint:
                ckpt_path = PROJECT_ROOT / "data" / f"ragas_checkpoint_{mode}{args.suffix}.json"
            ragas_scores = try_compute_ragas(rows, args.judge_provider, args.judge_model, ckpt_path, args.ragas_batch_size, args.ragas_max_retries)

        # Lưu chi tiết từng câu
        detail_path = PROJECT_ROOT / "data" / f"eval_scores_{mode}{args.suffix}.csv"
        with open(detail_path, "w", encoding="utf-8", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["id", "category", "completeness_rate", "response_preview"])
            for r in rows:
                writer.writerow([r["id"], r["category"], r.get("completeness_rate"), r["response"][:200].replace("\n", " ")])
        print(f"Đã lưu chi tiết: {detail_path}")

        # Tổng hợp theo category
        by_cat = defaultdict(list)
        for r in rows:
            if r.get("completeness_rate") is not None:
                by_cat[r["category"]].append(r["completeness_rate"])

        print(f"\n--- Legal Completeness Rate theo nhóm (mode={mode}) ---")
        for cat, vals in by_cat.items():
            avg = sum(vals) / len(vals) if vals else 0
            print(f"  {cat}: {avg:.2%} (n={len(vals)})")
            summary_rows.append({"mode": mode, "category": cat, "metric": "completeness_rate", "value": avg, "n": len(vals)})

        all_vals = [r["completeness_rate"] for r in rows if r.get("completeness_rate") is not None]
        overall = sum(all_vals) / len(all_vals) if all_vals else 0
        print(f"  TỔNG: {overall:.2%} (n={len(all_vals)})")
        summary_rows.append({"mode": mode, "category": "ALL", "metric": "completeness_rate", "value": overall, "n": len(all_vals)})

        # Chi phí token (prompt+completion cộng dồn qua MỌI lệnh gọi LLM trong 1 câu
        # hỏi — xem token_usage ghi bởi run_evaluation.py) — so sánh chi phí thực tế
        # giữa 3 kịch bản, không chỉ completeness_rate.
        token_totals = [r["token_usage"]["total_tokens"] for r in rows if r.get("token_usage")]
        prompt_totals = [r["token_usage"]["prompt_tokens"] for r in rows if r.get("token_usage")]
        completion_totals = [r["token_usage"]["completion_tokens"] for r in rows if r.get("token_usage")]
        call_counts = [r["token_usage"]["call_count"] for r in rows if r.get("token_usage")]
        if token_totals:
            avg_total = sum(token_totals) / len(token_totals)
            avg_prompt = sum(prompt_totals) / len(prompt_totals)
            avg_completion = sum(completion_totals) / len(completion_totals)
            avg_calls = sum(call_counts) / len(call_counts)
            print(f"  Token TB/câu: total={avg_total:.0f} (prompt={avg_prompt:.0f}, completion={avg_completion:.0f}), "
                  f"số lệnh gọi LLM TB={avg_calls:.1f} (n={len(token_totals)})")
            summary_rows.append({"mode": mode, "category": "ALL", "metric": "avg_total_tokens", "value": avg_total, "n": len(token_totals)})
            summary_rows.append({"mode": mode, "category": "ALL", "metric": "avg_prompt_tokens", "value": avg_prompt, "n": len(token_totals)})
            summary_rows.append({"mode": mode, "category": "ALL", "metric": "avg_completion_tokens", "value": avg_completion, "n": len(token_totals)})
            summary_rows.append({"mode": mode, "category": "ALL", "metric": "avg_llm_calls", "value": avg_calls, "n": len(token_totals)})

        # Token của ĐÚNG lệnh gọi sinh câu trả lời cuối cùng (không cộng router/gate/
        # draft-bị-bỏ) — chỉ số đúng cho "hiệu quả ngữ cảnh" khi so sánh 3 kịch bản,
        # tách biệt khỏi avg_total_tokens (tổng chi phí cả pipeline) ở trên.
        final_totals = [r["final_answer_token_usage"]["total_tokens"] for r in rows if r.get("final_answer_token_usage")]
        final_prompt = [r["final_answer_token_usage"]["prompt_tokens"] for r in rows if r.get("final_answer_token_usage")]
        final_completion = [r["final_answer_token_usage"]["completion_tokens"] for r in rows if r.get("final_answer_token_usage")]
        if final_totals:
            avg_final_total = sum(final_totals) / len(final_totals)
            avg_final_prompt = sum(final_prompt) / len(final_prompt)
            avg_final_completion = sum(final_completion) / len(final_completion)
            print(f"  Token TB/câu (CHỈ lệnh gọi sinh câu trả lời cuối): total={avg_final_total:.0f} "
                  f"(prompt={avg_final_prompt:.0f}, completion={avg_final_completion:.0f}) (n={len(final_totals)})")
            summary_rows.append({"mode": mode, "category": "ALL", "metric": "avg_final_answer_tokens", "value": avg_final_total, "n": len(final_totals)})
            summary_rows.append({"mode": mode, "category": "ALL", "metric": "avg_final_answer_prompt_tokens", "value": avg_final_prompt, "n": len(final_totals)})
            summary_rows.append({"mode": mode, "category": "ALL", "metric": "avg_final_answer_completion_tokens", "value": avg_final_completion, "n": len(final_totals)})

        if ragas_scores:
            print(f"\n--- RAGAS (mode={mode}) ---")
            for k, v in ragas_scores.items():
                label = RAGAS_METRIC_LABELS.get(k, k)
                print(f"  {label}: {v:.3f}")
                summary_rows.append({"mode": mode, "category": "ALL", "metric": f"ragas_{k}", "value": v, "n": len(rows)})

    summary_path = PROJECT_ROOT / "data" / f"eval_summary{args.suffix}.csv"
    with open(summary_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["mode", "category", "metric", "value", "n"])
        writer.writeheader()
        writer.writerows(summary_rows)
    print(f"\nĐã lưu bảng tổng hợp: {summary_path}")


if __name__ == "__main__":
    main()


## 8. Chấm điểm (Completeness Rate + RAGAS)

In [ ]:
modes_str = " ".join(MODES)
cmd = f'python scripts/score_evaluation.py --modes {modes_str}'
if OUTPUT_SUFFIX:
    cmd += f' --suffix "{OUTPUT_SUFFIX}"'
cmd += f' --judge-provider {JUDGE_PROVIDER}'
if JUDGE_MODEL:
    cmd += f' --judge-model {JUDGE_MODEL}'
if SKIP_RAGAS:
    cmd += ' --skip-ragas'
else:
    cmd += f' --ragas-batch-size {RAGAS_BATCH_SIZE} --ragas-max-retries {RAGAS_MAX_RETRIES}'
get_ipython().system(cmd)


## 9. Tính MRR / nDCG@5 / Context Precision-Recall / Avg context chars

In [ ]:
import json, math

def load_rows(mode):
    path = f"data/eval_results_{mode}{OUTPUT_SUFFIX}.jsonl"
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def mrr_ndcg(rows):
    sum_rr, sum_ndcg = 0.0, 0.0
    for r in rows:
        gt = {x.lower() for x in r["dieu_ids"]}
        ranked = [x.lower() for x in r["retrieved_dieu_ids_ranked"]]
        rank = next((i + 1 for i, d in enumerate(ranked) if d in gt), 0)
        sum_rr += 1 / rank if rank else 0
        dcg = sum(1 / math.log2(i + 2) for i, d in enumerate(ranked[:5]) if d in gt)
        ideal_hits = min(len(gt), 5)
        idcg = sum(1 / math.log2(i + 2) for i in range(ideal_hits))
        sum_ndcg += dcg / idcg if idcg > 0 else 0
    n = len(rows) or 1
    return sum_rr / n, sum_ndcg / n

def precision_recall(rows):
    precisions, recalls = [], []
    for r in rows:
        gt = {x.lower() for x in r["dieu_ids"]}
        final_set = {x.lower() for x in r["retrieved_dieu_ids_ranked"]} | {x.lower() for x in r["graph_fetched_dieu_ids"]}
        hit = len(final_set & gt)
        precisions.append(hit / len(final_set) if final_set else 0)
        recalls.append(min(hit, len(gt)) / len(gt) if gt else 0)
    n = len(rows) or 1
    return sum(precisions) / n, sum(recalls) / n

def avg_context_chars(rows):
    total = sum(sum(len(c) for c in (r.get("retrieved_contexts") or [])) for r in rows)
    return total / (len(rows) or 1)

# retrieved_dieu_ids_ranked giống hệt nhau ở cả 3 mode (bước retrieval dùng
# chung) — lấy mode ĐẦU TIÊN trong MODES thay vì cứng "naive", để vẫn chạy
# đúng nếu bạn bỏ "naive" ra khỏi danh sách MODES ở cell tham số.
mrr, ndcg = mrr_ndcg(load_rows(MODES[0]))
print(f"MRR — Mean Reciprocal Rank (retrieval chung cho cả {len(MODES)} mode) = {mrr:.3f}")
print(f"nDCG@5 — Normalized Discounted Cumulative Gain @5                  = {ndcg:.3f}\n")

retrieval_metrics = {}
for mode in MODES:
    rows = load_rows(mode)
    prec, rec = precision_recall(rows)
    chars = avg_context_chars(rows)
    retrieval_metrics[mode] = {"precision": prec, "recall": rec, "avg_context_chars": chars}
    print(f"[{mode}]")
    print(f"  Context Precision (theo Điều) = {prec:.3f}")
    print(f"  Context Recall (theo Điều)    = {rec:.3f}")
    print(f"  Avg context chars             = {chars:.0f}")


## 10. Bảng tổng hợp cuối cùng

In [ ]:
import csv

summary_path = f"data/eval_summary{OUTPUT_SUFFIX}.csv"
by_mode = {}
with open(summary_path, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        by_mode.setdefault(row["mode"], {})[f"{row['category']}::{row['metric']}"] = row["value"]

def fmt_pct(v):
    try:
        return f"{float(v) * 100:.1f}%"
    except (TypeError, ValueError):
        return "-"

def fmt_num(v):
    try:
        return f"{float(v):.0f}"
    except (TypeError, ValueError):
        return "-"

# Tên hiển thị RÕ RÀNG cho từng chỉ số RAGAS (key thật trong CSV là "ragas_<ten_metric>")
# — khớp với RAGAS_METRIC_LABELS trong scripts/score_evaluation.py để nhất quán.
RAGAS_METRIC_LABELS = {
    "ragas_faithfulness": "Faithfulness (độ trung thực với ngữ cảnh)",
    "ragas_answer_relevancy": "Answer Relevancy (độ liên quan câu trả lời)",
    "ragas_context_precision": "Context Precision (RAGAS)",
    "ragas_answer_correctness": "Answer Correctness (độ đúng đắn câu trả lời)",
}

print(f"\n{'=' * 100}\nBẢNG TỔNG HỢP{OUTPUT_SUFFIX}\n{'=' * 100}")
print(f"MRR — Mean Reciprocal Rank = {mrr:.3f}")
print(f"nDCG@5 — Normalized Discounted Cumulative Gain @5 = {ndcg:.3f}")
print(f"(2 chỉ số trên đo bước retrieval, CHUNG cho cả {len(MODES)} mode)\n")

header = f"{'Chỉ số':<40}" + "".join(f"{m:<20}" for m in MODES)
print(header)
print("-" * len(header))

def print_row(label, get_value_fn):
    print(f"{label:<40}" + "".join(f"{get_value_fn(m):<20}" for m in MODES))

# --- Legal Completeness Rate (chỉ số trung tâm của khóa luận) ---
print_row("Completeness Rate", lambda m: fmt_pct(by_mode.get(m, {}).get("ALL::completeness_rate")))

# --- Chi phí token ---
print_row("Avg total tokens/câu (cả pipeline)", lambda m: fmt_num(by_mode.get(m, {}).get("ALL::avg_total_tokens")))
print_row("Avg tokens/câu (chỉ câu trả lời cuối)", lambda m: fmt_num(by_mode.get(m, {}).get("ALL::avg_final_answer_tokens")))
print_row("Avg số lệnh gọi LLM/câu", lambda m: fmt_num(by_mode.get(m, {}).get("ALL::avg_llm_calls")))

# --- Context Precision/Recall tự tính (theo Điều, KHÁC với RAGAS context_precision) ---
print_row("Context Precision (theo Điều)", lambda m: f"{retrieval_metrics[m]['precision']:.3f}")
print_row("Context Recall (theo Điều)", lambda m: f"{retrieval_metrics[m]['recall']:.3f}")
print_row("Avg context chars", lambda m: f"{retrieval_metrics[m]['avg_context_chars']:.0f}")

# --- RAGAS (nếu có chạy) — in TÊN RÕ RÀNG cho từng chỉ số, cùng bảng luôn ---
if not SKIP_RAGAS:
    print()
    print(f"--- RAGAS (provider={RAGAS_PROVIDER}{', model=' + RAGAS_MODEL if RAGAS_MODEL else ''}) ---")
    for ragas_key, label in RAGAS_METRIC_LABELS.items():
        print_row(label, lambda m, k=ragas_key: fmt_pct(by_mode.get(m, {}).get(f"ALL::{k}")))


## 11. Sao lưu kết quả về Google Drive (Colab mất hết khi hết phiên)

In [ ]:
import shutil, glob, os

dest = os.path.join(DRIVE_DATA_DIR, "results")
os.makedirs(dest, exist_ok=True)

patterns = [
    f"data/eval_results_*{OUTPUT_SUFFIX}.jsonl",
    f"data/eval_scores_*{OUTPUT_SUFFIX}.csv",
    f"data/eval_summary{OUTPUT_SUFFIX}.csv",
]
for pattern in patterns:
    for f in glob.glob(pattern):
        shutil.copy(f, dest)
        print(f"Đã copy {f} -> {dest}")

print("\nHoàn tất — kết quả đã lưu vào Google Drive, không mất khi hết phiên Colab.")
